In [1]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score,cross_val_predict
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold
from sklearn import metrics
import statsmodels.api as sm
from sklearn.linear_model import Lasso, Ridge


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
sns.set_context('paper')

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
train=pd.read_csv("/home/carlton/Downloads/abb_hackathon/train_v9rqX0R.csv")
test=pd.read_csv("/home/carlton/Downloads/abb_hackathon/test_AbJTz2l.csv")

## Data Analysis

In [4]:
train.Outlet_Size.value_counts()

Medium    2793
Small     2388
High       932
Name: Outlet_Size, dtype: int64

## Train  distribution of `Outlet_Size` across different `Outlet_Type` categories in the training dataset

In [5]:
crosstable = pd.crosstab(train['Outlet_Size'],train['Outlet_Type'])
crosstable

Outlet_Type,Grocery Store,Supermarket Type1,Supermarket Type2,Supermarket Type3
Outlet_Size,,,,
High,0,932,0,0
Medium,0,930,928,935
Small,528,1860,0,0


In [6]:
dic = {'Grocery Store':'Small'}
s = train.Outlet_Type.map(dic)

train.Outlet_Size= train.Outlet_Size.combine_first(s)
train.Outlet_Size.value_counts()

Small     2943
Medium    2793
High       932
Name: Outlet_Size, dtype: int64

# Test Train distribution of Outlet_Size across different Outlet_Type categories in the training dataset

In [7]:
crosstable = pd.crosstab(test['Outlet_Size'],test['Outlet_Type'])
crosstable

Outlet_Type,Grocery Store,Supermarket Type1,Supermarket Type2,Supermarket Type3
Outlet_Size,,,,
High,0,621,0,0
Medium,0,620,618,624
Small,352,1240,0,0


In [8]:
dic = {'Grocery Store':'Small'}
s = test.Outlet_Type.map(dic)

test.Outlet_Size= test.Outlet_Size.combine_first(s)
test.Outlet_Size.value_counts()

Small     1962
Medium    1862
High       621
Name: Outlet_Size, dtype: int64

In [9]:
train.isnull().sum(axis=0)

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  1855
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [10]:
test.isnull().sum(axis=0)

Item_Identifier                 0
Item_Weight                   976
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  1236
Outlet_Location_Type            0
Outlet_Type                     0
dtype: int64

### Distribution of Outlet Size across Outlet Location Type


In [11]:
train.Outlet_Size.value_counts()

Small     2943
Medium    2793
High       932
Name: Outlet_Size, dtype: int64

In [12]:
#checking for location type
crosstable = pd.crosstab(train.Outlet_Size,train.Outlet_Location_Type)
crosstable

Outlet_Location_Type,Tier 1,Tier 2,Tier 3
Outlet_Size,,,
High,0,0,932
Medium,930,0,1863
Small,1458,930,555


### Outlet Size Imputation Logic

Mapped all `Outlet_Location_Type == "Tier 2"` to `Outlet_Size = "Small"` for missing values, then displayed the updated counts of `Outlet_Size`.


In [13]:
dic = {"Tier 2":"Small"}
s = train.Outlet_Location_Type.map(dic)
train.Outlet_Size = train.Outlet_Size.combine_first(s)
train.Outlet_Size.value_counts()

Small     4798
Medium    2793
High       932
Name: Outlet_Size, dtype: int64

In [14]:
test.Outlet_Size.value_counts()

Small     1962
Medium    1862
High       621
Name: Outlet_Size, dtype: int64

In [15]:
#checking for location type
crosstable = pd.crosstab(test.Outlet_Size,test.Outlet_Location_Type)
crosstable

Outlet_Location_Type,Tier 1,Tier 2,Tier 3
Outlet_Size,,,
High,0,0,621
Medium,620,0,1242
Small,972,620,370


doubt

In [16]:
dic = {"Tier 2":"Small"}
s = test.Outlet_Location_Type.map(dic)
test.Outlet_Size = test.Outlet_Size.combine_first(s)
test.Outlet_Size.value_counts()

Small     3198
Medium    1862
High       621
Name: Outlet_Size, dtype: int64

### Imputing Outlet Size in Test Data Using Outlet Type

Filled missing `Outlet_Size` values in the test set using a predefined mapping from `Outlet_Type` to `Outlet_Size` based on distribution insights. Finally, printed the number of remaining missing values after imputation.


In [17]:
# Mapping based on your outlet_type vs size distribution
size_map = {
    'Grocery Store': 'Small',
    'Supermarket Type1': 'Small',
    'Supermarket Type2': 'Medium',
    'Supermarket Type3': 'Medium'
}

# Fill Outlet_Size using this mapping
test['Outlet_Size'] = test['Outlet_Size'].combine_first(
    test['Outlet_Type'].map(size_map)
)

# Optional: check missing after fill
print("Remaining NaNs:", test['Outlet_Size'].isna().sum())


Remaining NaNs: 0


In [18]:
train.isnull().sum(axis=0)

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                     0
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [19]:
test.isnull().sum(axis=0)

Item_Identifier                0
Item_Weight                  976
Item_Fat_Content               0
Item_Visibility                0
Item_Type                      0
Item_MRP                       0
Outlet_Identifier              0
Outlet_Establishment_Year      0
Outlet_Size                    0
Outlet_Location_Type           0
Outlet_Type                    0
dtype: int64

### Imputing Missing Item Weight in Training Data

Missing values in `Item_Weight` are filled using the mean weight per `Item_Identifier`, ensuring consistent and specific imputation. The remaining missing values are then summarized.


In [20]:
train['Item_Weight']=train['Item_Weight'].fillna(train.groupby('Item_Identifier')['Item_Weight'].transform('mean'))
train.isnull().sum()

Item_Identifier              0
Item_Weight                  4
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64

In [21]:
train[train.Item_Weight.isnull()]

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
927,FDN52,NaN,Regular,0.130933,Frozen Foods,86.9198,OUT027,1985,Medium,Tier 3,Supermarket Type3,1569.9564
1922,FDK57,NaN,Low Fat,0.079904,Snack Foods,120.0440,OUT027,1985,Medium,Tier 3,Supermarket Type3,4434.2280
4187,FDE52,NaN,Regular,0.029742,Dairy,88.9514,OUT027,1985,Medium,Tier 3,Supermarket Type3,3453.5046
5022,FDQ60,NaN,Regular,0.191501,Baking Goods,121.2098,OUT019,1985,Small,Tier 1,Grocery Store,120.5098


In [22]:
# List of item types 
item_type_list = train.Item_Type.unique().tolist()
item_type_list

['Dairy',
 'Soft Drinks',
 'Meat',
 'Fruits and Vegetables',
 'Household',
 'Baking Goods',
 'Snack Foods',
 'Frozen Foods',
 'Breakfast',
 'Health and Hygiene',
 'Hard Drinks',
 'Canned',
 'Breads',
 'Starchy Foods',
 'Others',
 'Seafood']

In [23]:
# grouping based on item type and calculating mean of item weight
Item_Type_Means = train.groupby('Item_Type')['Item_Weight'].mean() 

In [24]:
Item_Type_Means

Item_Type
Baking Goods             12.285317
Breads                   11.297689
Breakfast                12.779727
Canned                   12.403320
Dairy                    13.379905
Frozen Foods             12.782404
Fruits and Vegetables    13.236713
Hard Drinks              11.456238
Health and Hygiene       13.052327
Household                13.524780
Meat                     12.771212
Others                   13.979438
Seafood                  12.521953
Snack Foods              13.031230
Soft Drinks              11.879775
Starchy Foods            13.841385
Name: Item_Weight, dtype: float64

### Imputing Missing Item Weight Using Item Type Means

For each `Item_Type`, missing `Item_Weight` values are filled with the corresponding mean from `Item_Type_Means`. This ensures more context-aware imputation rather than using a global average.


In [25]:
# Mapiing Item weight to item type mean
for i in item_type_list:
    dic = {i:Item_Type_Means[i]}
    s = train.Item_Type.map(dic)
    train.Item_Weight = train.Item_Weight.combine_first(s)
    
Item_Type_Means = train.groupby('Item_Type')['Item_Weight'].mean() 

train.isnull().sum()


Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64

In [ ]:
### Imputing Missing Item Weight in Test Data

Filled missing `Item_Weight` values in the test set using the mean weight of each `Item_Identifier`. This preserves item-specific consistency. Displayed the count of remaining missing values after imputation.


In [ ]:
test['Item_Weight']=test['Item_Weight'].fillna(test.groupby('Item_Identifier')['Item_Weight'].transform('mean'))
test.isnull().sum()

In [27]:
# List of item types 
item_type_list = test.Item_Type.unique().tolist()
item_type_list

['Snack Foods',
 'Dairy',
 'Others',
 'Fruits and Vegetables',
 'Baking Goods',
 'Health and Hygiene',
 'Breads',
 'Hard Drinks',
 'Seafood',
 'Soft Drinks',
 'Household',
 'Frozen Foods',
 'Meat',
 'Canned',
 'Starchy Foods',
 'Breakfast']

In [28]:
# grouping based on item type and calculating mean of item weight
Item_Type_Means = test.groupby('Item_Type')['Item_Weight'].mean() 

In [29]:
# Mapiing Item weight to item type mean
for i in item_type_list:
    dic = {i:Item_Type_Means[i]}
    s = test.Item_Type.map(dic)
    test.Item_Weight = test.Item_Weight.combine_first(s)
    
Item_Type_Means = test.groupby('Item_Type')['Item_Weight'].mean() 

test.isnull().sum()


Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
dtype: int64

In [30]:
test[test.Item_Weight.isnull()]

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type


In [31]:
train.Item_Visibility.value_counts().head() 

0.000000    526
0.076975      3
0.041283      2
0.085622      2
0.187841      2
Name: Item_Visibility, dtype: int64

In [32]:
import numpy as np
# Replacing 0's with NaN
train.Item_Visibility.replace(to_replace=0.000000,value=np.NaN,inplace=True)
# Now fill by mean of visbility based on item identifiers
train.Item_Visibility = train.Item_Visibility.fillna(train.groupby('Item_Identifier')['Item_Visibility'].transform('mean'))


In [33]:
train.Item_Visibility.value_counts().head()

0.121880    4
0.082138    3
0.016164    3
0.029511    3
0.081428    3
Name: Item_Visibility, dtype: int64

In [34]:
test.Item_Visibility.value_counts().head() 

0.000000    353
0.107493      2
0.159518      2
0.026711      2
0.056306      2
Name: Item_Visibility, dtype: int64

In [35]:
import numpy as np
# Replacing 0's with NaN
test.Item_Visibility.replace(to_replace=0.000000,value=np.NaN,inplace=True)
# Now fill by mean of visbility based on item identifiers
test.Item_Visibility = test.Item_Visibility.fillna(test.groupby('Item_Identifier')['Item_Visibility'].transform('mean'))


In [36]:
test.Item_Visibility.value_counts().head()

0.012002    3
0.065787    3
0.127416    3
0.072816    3
0.017073    2
Name: Item_Visibility, dtype: int64

In [37]:
test['Item_Visibility'].fillna(test['Item_Visibility'].mean(), inplace=True)


In [38]:
# test['Item_Visibility'] = test['Item_Visibility'].fillna(
#     test.groupby('Item_Identifier')['Item_Visibility'].transform('mean')
# )


In [39]:

test.isnull().sum()

Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
dtype: int64

In [40]:
train.Item_Fat_Content.value_counts()

Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: Item_Fat_Content, dtype: int64

In [41]:
train.Item_Fat_Content.replace(to_replace=["LF","low fat"],value="Low Fat",inplace=True)
train.Item_Fat_Content.replace(to_replace="reg",value="Regular",inplace=True)

train.Item_Fat_Content.value_counts()

Low Fat    5517
Regular    3006
Name: Item_Fat_Content, dtype: int64

In [42]:
test.Item_Fat_Content.value_counts()

Low Fat    3396
Regular    1935
LF          206
reg          78
low fat      66
Name: Item_Fat_Content, dtype: int64

In [43]:
test.Item_Fat_Content.replace(to_replace=["LF","low fat"],value="Low Fat",inplace=True)
test.Item_Fat_Content.replace(to_replace="reg",value="Regular",inplace=True)

test.Item_Fat_Content.value_counts()

Low Fat    3668
Regular    2013
Name: Item_Fat_Content, dtype: int64

In [44]:
train['Outlet_Year'] = (2013 - train.Outlet_Establishment_Year)

In [45]:
test['Outlet_Year'] = (2013 - test.Outlet_Establishment_Year)

In [46]:
var_cat = train.select_dtypes(include=[object])
var_cat.head()

,Item_Identifier,Item_Fat_Content,Item_Type,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Outlet_Type
0,FDA15,Low Fat,Dairy,OUT049,Medium,Tier 1,Supermarket Type1
1,DRC01,Regular,Soft Drinks,OUT018,Medium,Tier 3,Supermarket Type2
2,FDN15,Low Fat,Meat,OUT049,Medium,Tier 1,Supermarket Type1
3,FDX07,Regular,Fruits and Vegetables,OUT010,Small,Tier 3,Grocery Store
4,NCD19,Low Fat,Household,OUT013,High,Tier 3,Supermarket Type1


In [47]:
#Convert categorical into numerical 
var_cat = var_cat.columns.tolist()
var_cat = ['Item_Fat_Content',
 'Item_Type',
 'Outlet_Size',
 'Outlet_Location_Type',
 'Outlet_Type']

var_cat

['Item_Fat_Content',
 'Item_Type',
 'Outlet_Size',
 'Outlet_Location_Type',
 'Outlet_Type']

In [48]:
train['Item_Type_New'] = train.Item_Identifier
train.Item_Type_New.head(10)

0    FDA15
1    DRC01
2    FDN15
3    FDX07
4    NCD19
5    FDP36
6    FDO10
7    FDP10
8    FDH17
9    FDU28
Name: Item_Type_New, dtype: object

In [49]:
train.Item_Type_New.replace(to_replace="^FD*.*",value="Food",regex=True,inplace=True)
train.Item_Type_New.replace(to_replace="^DR*.*",value="Drinks",regex=True,inplace=True)
train.Item_Type_New.replace(to_replace="^NC*.*",value="Non-Consumable",regex=True,inplace=True)

train.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Outlet_Year,Item_Type_New
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,14,Food
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,4,Drinks
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,14,Food
3,FDX07,19.20,Regular,0.022911,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,15,Food
4,NCD19,8.93,Low Fat,0.016164,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,26,Non-Consumable


In [50]:
test

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Outlet_Year
0,FDW58,20.750,Low Fat,0.007565,Snack Foods,107.8622,OUT049,1999,Medium,Tier 1,Supermarket Type1,14
1,FDW14,8.300,Regular,0.038428,Dairy,87.3198,OUT017,2007,Small,Tier 2,Supermarket Type1,6
2,NCN55,14.600,Low Fat,0.099575,Others,241.7538,OUT010,1998,Small,Tier 3,Grocery Store,15
3,FDQ58,7.315,Low Fat,0.015388,Snack Foods,155.0340,OUT017,2007,Small,Tier 2,Supermarket Type1,6
4,FDY38,13.600,Regular,0.118599,Dairy,234.2300,OUT027,1985,Medium,Tier 3,Supermarket Type3,28
5,FDH56,9.800,Regular,0.063817,Fruits and Vegetables,117.1492,OUT046,1997,Small,Tier 1,Supermarket Type1,16
6,FDL48,19.350,Regular,0.082602,Baking Goods,50.1034,OUT018,2009,Medium,Tier 3,Supermarket Type2,4
7,FDC48,9.195,Low Fat,0.015782,Baking Goods,81.0592,OUT027,1985,Medium,Tier 3,Supermarket Type3,28
8,FDN33,6.305,Regular,0.123365,Snack Foods,95.7436,OUT045,2002,Small,Tier 2,Supermarket Type1,11
9,FDA36,5.985,Low Fat,0.005698,Baking Goods,186.8924,OUT017,2007,Small,Tier 2,Supermarket Type1,6


In [51]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le_outlet = LabelEncoder()
le_item = LabelEncoder()

In [52]:
train['Outlet'] = le_outlet.fit_transform(train.Outlet_Identifier)
train['Item'] = le_item.fit_transform(train.Item_Type_New)
train.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Outlet_Year,Item_Type_New,Outlet,Item
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,14,Food,9,1
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,4,Drinks,3,0
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,14,Food,9,1
3,FDX07,19.20,Regular,0.022911,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,15,Food,0,1
4,NCD19,8.93,Low Fat,0.016164,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,26,Non-Consumable,1,2


In [53]:
from sklearn.preprocessing import LabelEncoder

le_fat = LabelEncoder()
train['Item_Fat_Content'] = le_fat.fit_transform(train['Item_Fat_Content'].astype(str))

le_type = LabelEncoder()
train['Item_Type'] = le_type.fit_transform(train['Item_Type'].astype(str))

le_size = LabelEncoder()
train['Outlet_Size'] = le_size.fit_transform(train['Outlet_Size'].astype(str))

le_location = LabelEncoder()
train['Outlet_Location_Type'] = le_location.fit_transform(train['Outlet_Location_Type'].astype(str))

le_otype = LabelEncoder()
train['Outlet_Type'] = le_otype.fit_transform(train['Outlet_Type'].astype(str))


In [54]:
test['Item_Type_New'] = test.Item_Identifier
test.Item_Type_New.head(10)

0    FDW58
1    FDW14
2    NCN55
3    FDQ58
4    FDY38
5    FDH56
6    FDL48
7    FDC48
8    FDN33
9    FDA36
Name: Item_Type_New, dtype: object

In [55]:
test.Item_Type_New.replace(to_replace="^FD*.*",value="Food",regex=True,inplace=True)
test.Item_Type_New.replace(to_replace="^DR*.*",value="Drinks",regex=True,inplace=True)
test.Item_Type_New.replace(to_replace="^NC*.*",value="Non-Consumable",regex=True,inplace=True)

test.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Outlet_Year,Item_Type_New
0,FDW58,20.750,Low Fat,0.007565,Snack Foods,107.8622,OUT049,1999,Medium,Tier 1,Supermarket Type1,14,Food
1,FDW14,8.300,Regular,0.038428,Dairy,87.3198,OUT017,2007,Small,Tier 2,Supermarket Type1,6,Food
2,NCN55,14.600,Low Fat,0.099575,Others,241.7538,OUT010,1998,Small,Tier 3,Grocery Store,15,Non-Consumable
3,FDQ58,7.315,Low Fat,0.015388,Snack Foods,155.0340,OUT017,2007,Small,Tier 2,Supermarket Type1,6,Food
4,FDY38,13.600,Regular,0.118599,Dairy,234.2300,OUT027,1985,Medium,Tier 3,Supermarket Type3,28,Food


In [56]:
def safe_label_transform(le, values, default=-1):
    known = set(le.classes_)
    return [le.transform([v])[0] if v in known else default for v in values]

test['Outlet'] = safe_label_transform(le_outlet, test['Outlet_Identifier'])
test['Item'] = safe_label_transform(le_item, test['Item_Type_New'])


In [57]:
# Safe transform for unseen labels
def safe_label_transform(le, values, default=-1):
    known = set(le.classes_)
    return [le.transform([v])[0] if v in known else default for v in values]

# Apply train-fitted encoders to test data
test['Item_Fat_Content'] = safe_label_transform(le_fat, test['Item_Fat_Content'].astype(str))
test['Item_Type'] = safe_label_transform(le_type, test['Item_Type'].astype(str))
test['Outlet_Size'] = safe_label_transform(le_size, test['Outlet_Size'].astype(str))
test['Outlet_Location_Type'] = safe_label_transform(le_location, test['Outlet_Location_Type'].astype(str))
test['Outlet_Type'] = safe_label_transform(le_otype, test['Outlet_Type'].astype(str))


In [58]:
#Visualizing Correlation
corrmat = train.corr()
corrmat
# corrmat.to_csv("original_.csv")

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Outlet_Year,Outlet,Item
Item_Weight,1.000000,-0.026797,-0.021226,0.035710,0.025967,-0.013417,-0.012445,0.002973,0.000592,0.013198,0.013417,-0.007586,0.071585
Item_Fat_Content,-0.026797,1.000000,0.052620,-0.139434,0.006063,0.003151,-0.001262,-0.001598,0.002199,0.018719,-0.003151,0.000764,-0.166111
Item_Visibility,-0.021226,0.052620,1.000000,-0.042400,-0.005951,-0.075238,0.090322,-0.030449,-0.170752,-0.126026,0.075238,-0.096282,-0.050520
Item_Type,0.035710,-0.139434,-0.042400,1.000000,0.032651,0.004970,-0.000218,0.003084,0.003053,0.017048,-0.004970,0.001656,0.007456
Item_MRP,0.025967,0.006063,-0.005951,0.032651,1.000000,0.005020,0.000872,0.000232,-0.001975,0.567574,-0.005020,0.003319,0.032517
Outlet_Establishment_Year,-0.013417,0.003151,-0.075238,0.004970,0.005020,1.000000,0.425534,-0.089216,-0.122304,-0.049135,-1.000000,0.079035,-0.008551
Outlet_Size,-0.012445,-0.001262,0.090322,-0.000218,0.000872,0.425534,1.000000,-0.480075,-0.401373,-0.162753,-0.425534,0.260272,-0.001276
Outlet_Location_Type,0.002973,-0.001598,-0.030449,0.003084,0.000232,-0.089216,-0.480075,1.000000,0.467219,0.089367,0.089216,-0.716176,0.007661
Outlet_Type,0.000592,0.002199,-0.170752,0.003053,-0.001975,-0.122304,-0.401373,0.467219,1.000000,0.401522,0.122304,0.099873,0.001136
Item_Outlet_Sales,0.013198,0.018719,-0.126026,0.017048,0.567574,-0.049135,-0.162753,0.089367,0.401522,1.000000,0.049135,0.162325,0.011236


# Predictive Modelling

In [59]:
seed = 240
np.random.seed(seed)

In [60]:
predictors=['Item_MRP','Outlet_Size','Outlet_Location_Type','Outlet_Type','Outlet_Year']

In [61]:
extra_features = [col for col in test.columns if col not in predictors]
print("🟨 Columns in train but not in predictors:", extra_features)

🟨 Columns in train but not in predictors: ['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Outlet_Identifier', 'Outlet_Establishment_Year', 'Item_Type_New', 'Outlet', 'Item']


In [62]:
train['Outlet_NM'] = train['Outlet_Identifier'].str[-2:].astype(int)
test['Outlet_NM'] = test['Outlet_Identifier'].str[-2:].astype(int)

In [63]:
def add_item_category_onehot(train, test):
    # Step 1: Extract category
    train['Item_Category'] = train['Item_Identifier'].str[:2]
    test['Item_Category'] = test['Item_Identifier'].str[:2]

    # Step 2: One-hot encode
    train_dummies = pd.get_dummies(train['Item_Category'], prefix='ItemCat')
    test_dummies = pd.get_dummies(test['Item_Category'], prefix='ItemCat')

    # Step 3: Align columns
    train_dummies, test_dummies = train_dummies.align(test_dummies, join='outer', axis=1, fill_value=0)

    # Step 4: Append to original DataFrames
    train = pd.concat([train, train_dummies], axis=1)
    test = pd.concat([test, test_dummies], axis=1)

    # Optional: Drop the intermediate column
#     train.drop(columns='Item_Category', inplace=True)
#     test.drop(columns='Item_Category', inplace=True)

    return train, test

In [64]:
def encode_item_category(train, test):
    mapping = {'FD': 1, 'NC': 2, 'DR': 3}
    train['Item_Category_Code'] = train['Item_Identifier'].str[:2].map(mapping)
    test['Item_Category_Code'] = test['Item_Identifier'].str[:2].map(mapping)
    return train, test

In [65]:
train, test = encode_item_category(train, test)

In [66]:
train, test = add_item_category_onehot(train, test)

In [67]:
train[extra_features]

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Outlet_Identifier,Outlet_Establishment_Year,Item_Type_New,Outlet,Item
0,FDA15,9.300,0,0.016047,4,OUT049,1999,Food,9,1
1,DRC01,5.920,1,0.019278,14,OUT018,2009,Drinks,3,0
2,FDN15,17.500,0,0.016760,10,OUT049,1999,Food,9,1
3,FDX07,19.200,1,0.022911,6,OUT010,1998,Food,0,1
4,NCD19,8.930,0,0.016164,9,OUT013,1987,Non-Consumable,1,2
5,FDP36,10.395,1,0.091392,0,OUT018,2009,Food,3,1
6,FDO10,13.650,1,0.012741,13,OUT013,1987,Food,1,1
7,FDP10,19.000,0,0.127470,13,OUT027,1985,Food,5,1
8,FDH17,16.200,1,0.016687,5,OUT045,2002,Food,7,1
9,FDU28,19.200,1,0.094450,5,OUT017,2007,Food,2,1


In [68]:
X = train[predictors]
y = train.Item_Outlet_Sales

In [69]:
# X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.25,random_state = 42)

In [70]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
     # must be a column in X
)


In [71]:
X_train.shape

(6392, 5)

In [1134]:
def evaluate_and_predict_final(X_train, X_test, y_train, y_test, test_final, submission_file="submission.csv"):
    """
    Evaluates models on train/test split.
    Uses best model to predict on separate test_final (no target).
    test_final must include Item_Identifier & Outlet_Identifier.
    Saves predictions to CSV.
    """
    from sklearn.linear_model import LinearRegression, Ridge, Lasso
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    from sklearn.neural_network import MLPRegressor
    from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
    import pandas as pd

    models = {
        'LinearRegression': LinearRegression(),
        'Ridge': Ridge(),
        'Lasso': Lasso(),
        'DecisionTree': DecisionTreeRegressor(random_state=42),
        'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
        'GradientBoosting': GradientBoostingRegressor( random_state=42),
        'MLPRegressor': MLPRegressor(hidden_layer_sizes=(1024, 512, 256, 128), max_iter=500, random_state=42)
    }

    results = []
    model_store = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        rmse = mean_squared_error(y_test, preds, squared=False)

        results.append({
            'Model': name,
            'MAE': round(mean_absolute_error(y_test, preds), 2),
            'MAPE': round(mean_absolute_percentage_error(y_test, preds) * 100, 2),
            'RMSE': round(rmse, 2),
            'R2 Score': round(r2_score(y_test, preds), 4)
        })

        model_store[name] = model

    results_df = pd.DataFrame(results)
    best_model_name = results_df.sort_values(by='RMSE').iloc[0]['Model']
    best_model = model_store[best_model_name]

    print(f"✅ Best model: {best_model_name} (RMSE: {results_df.loc[results_df['Model'] == best_model_name, 'RMSE'].values[0]})")

    # Drop ID cols before prediction

    test_preds = best_model.predict(test_final)

    # Save submission
    submission = pd.DataFrame({
        'Item_Identifier': test['Item_Identifier'],
        'Outlet_Identifier': test['Outlet_Identifier'],
        'Item_Outlet_Sales': np.clip(test_preds, 0, None)
    })

    submission.to_csv(submission_file, index=False)
    print(f"📁 Submission saved to: {submission_file}")

    return results_df


In [1131]:
test_final=test[predictors]

In [1132]:
results = evaluate_and_predict_final(X_train, X_test, y_train, y_test, test_final=test_final)


✅ Best model: MLPRegressor (RMSE: 1067.28)
📁 Submission saved to: submission.csv


In [1133]:
results

,MAE,MAPE,Model,R2 Score,RMSE
0,870.56,103.86,LinearRegression,0.5052,1168.38
1,870.46,103.81,Ridge,0.5052,1168.35
2,870.34,103.62,Lasso,0.5054,1168.21
3,1007.43,70.94,DecisionTree,0.2433,1444.88
4,763.88,58.71,RandomForest,0.5625,1098.68
5,841.90,69.25,GradientBoosting,0.4789,1199.08
6,748.59,62.47,MLPRegressor,0.5871,1067.28


## 700

In [192]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

class TorchMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 2048), nn.ReLU(),
            nn.Linear(2048, 1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)

def evaluate_and_predict_final(X_train, X_test, y_train, y_test, test_final, test_ids, submission_file="submission.csv"):
    """
    Trains PyTorch MLP on X_train/y_train, evaluates on X_test/y_test.
    Predicts on test_final and saves predictions to CSV with test_ids.
    """

    # Ensure numpy
    X_train = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
    X_test = X_test.values if isinstance(X_test, pd.DataFrame) else X_test
    y_train = y_train.values if isinstance(y_train, pd.Series) else y_train
    y_test = y_test.values if isinstance(y_test, pd.Series) else y_test
    test_final = test_final.values if isinstance(test_final, pd.DataFrame) else test_final

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)

    model = TorchMLP(X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("🔥 Training PyTorch MLP...")
    for epoch in range(1, 2000):
        model.train()
        optimizer.zero_grad()
        output = model(X_train_tensor)
        loss = criterion(output, y_train_tensor)
        loss.backward()
        optimizer.step()

        if epoch == 1 or epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                val_preds = model(X_test_tensor).cpu().numpy().flatten()
                val_true = y_test_tensor.cpu().numpy().flatten()
                rmse = mean_squared_error(val_true, val_preds, squared=False)
                r2 = r2_score(val_true, val_preds)
                mape = mean_absolute_percentage_error(val_true, val_preds) * 100

            print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | RMSE: {rmse:.2f} | R²: {r2:.4f} | MAPE: {mape:.2f}%")

    # Prediction on test_final
    model.eval()
    X_final_tensor = torch.tensor(test_final, dtype=torch.float32).to(device)
    with torch.no_grad():
        final_preds = model(X_final_tensor).cpu().numpy().flatten()
        final_preds = np.clip(final_preds, 0, None)

    # Save submission
    submission = pd.DataFrame({
        'Item_Identifier': test_ids['Item_Identifier'].values,
        'Outlet_Identifier': test_ids['Outlet_Identifier'].values,
        'Item_Outlet_Sales': final_preds
    })

    submission.to_csv(submission_file, index=False)
    print(f"✅ Submission saved to: {submission_file}")


In [193]:
evaluate_and_predict_final(
    X_train, X_test, y_train, y_test,
    test_final=test[predictors],
    test_ids=test[['Item_Identifier', 'Outlet_Identifier']]
)


🔥 Training PyTorch MLP...
Epoch 001 | Loss: 6284672.0000 | RMSE: 2530.58 | R²: -1.8935 | MAPE: 99.85%
Epoch 010 | Loss: 3340672.5000 | RMSE: 1512.85 | R²: -0.0341 | MAPE: 93.44%
Epoch 020 | Loss: 2068844.1250 | RMSE: 1456.12 | R²: 0.0420 | MAPE: 96.94%
Epoch 030 | Loss: 1646004.5000 | RMSE: 1233.25 | R²: 0.3128 | MAPE: 142.88%
Epoch 040 | Loss: 1529270.1250 | RMSE: 1236.87 | R²: 0.3088 | MAPE: 145.91%
Epoch 050 | Loss: 1542834.8750 | RMSE: 1233.56 | R²: 0.3124 | MAPE: 126.95%
Epoch 060 | Loss: 1514433.5000 | RMSE: 1228.96 | R²: 0.3176 | MAPE: 128.25%
Epoch 070 | Loss: 1503066.1250 | RMSE: 1220.29 | R²: 0.3272 | MAPE: 133.60%
Epoch 080 | Loss: 1494068.6250 | RMSE: 1216.64 | R²: 0.3312 | MAPE: 135.42%
Epoch 090 | Loss: 1482726.3750 | RMSE: 1213.24 | R²: 0.3349 | MAPE: 135.05%
Epoch 100 | Loss: 1471952.2500 | RMSE: 1209.94 | R²: 0.3385 | MAPE: 134.49%
Epoch 110 | Loss: 1460177.6250 | RMSE: 1206.36 | R²: 0.3424 | MAPE: 133.84%
Epoch 120 | Loss: 1445644.7500 | RMSE: 1201.89 | R²: 0.3473 | M

Epoch 1100 | Loss: 1134294.5000 | RMSE: 1075.46 | R²: 0.4774 | MAPE: 107.89%
Epoch 1110 | Loss: 1119472.2500 | RMSE: 1062.87 | R²: 0.4896 | MAPE: 92.45%
Epoch 1120 | Loss: 1101194.1250 | RMSE: 1055.02 | R²: 0.4971 | MAPE: 88.10%
Epoch 1130 | Loss: 1088221.6250 | RMSE: 1046.89 | R²: 0.5048 | MAPE: 86.34%
Epoch 1140 | Loss: 1079073.6250 | RMSE: 1040.87 | R²: 0.5105 | MAPE: 83.43%
Epoch 1150 | Loss: 1072185.2500 | RMSE: 1036.52 | R²: 0.5145 | MAPE: 79.59%
Epoch 1160 | Loss: 1063930.0000 | RMSE: 1031.70 | R²: 0.5191 | MAPE: 76.19%
Epoch 1170 | Loss: 1041651.0000 | RMSE: 1019.47 | R²: 0.5304 | MAPE: 69.80%
Epoch 1180 | Loss: 1018089.9375 | RMSE: 1006.80 | R²: 0.5420 | MAPE: 60.57%
Epoch 1190 | Loss: 985911.9375 | RMSE: 990.18 | R²: 0.5570 | MAPE: 58.30%
Epoch 1200 | Loss: 964674.1875 | RMSE: 981.69 | R²: 0.5645 | MAPE: 57.67%
Epoch 1210 | Loss: 953130.5625 | RMSE: 976.97 | R²: 0.5687 | MAPE: 57.95%
Epoch 1220 | Loss: 947768.2500 | RMSE: 975.27 | R²: 0.5702 | MAPE: 55.10%
Epoch 1230 | Loss: 

In [136]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

# ---------------------- MLP Definition ----------------------
import torch.nn as nn

class TorchMLP(nn.Module):
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(),

            nn.Linear(4096, 2048),
            nn.ReLU(),

            nn.Linear(2048, 1024),
            nn.ReLU(),

            nn.Linear(1024, 768),
            nn.ReLU(),

            nn.Linear(768, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),  # Dropout added here (deeper stage)

            nn.Linear(256, 128),
            nn.ReLU(),

    
            nn.Linear(128, 64),
            nn.Dropout(dropout_rate), 
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)



# ---------------------- Training + Prediction ----------------------
def evaluate_and_predict_final(X_train, X_test, y_train, y_test, test_final, test_ids, submission_file="submission.csv", model_path="best_model.pt"):
    """
    Trains PyTorch MLP on X_train/y_train, evaluates on X_test/y_test.
    Saves best model (lowest RMSE) to model_path.
    Predicts on test_final and saves submission to CSV with test_ids.
    """
    # Ensure numpy
    X_train = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
    X_test = X_test.values if isinstance(X_test, pd.DataFrame) else X_test
    y_train = y_train.values if isinstance(y_train, pd.Series) else y_train
    y_test = y_test.values if isinstance(y_test, pd.Series) else y_test
    test_final = test_final.values if isinstance(test_final, pd.DataFrame) else test_final

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)

    model = TorchMLP(X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_rmse = float('inf')

    print("🔥 Training PyTorch MLP...")
    for epoch in range(1, 2000):
        model.train()
        optimizer.zero_grad()
        output = model(X_train_tensor)
        loss = criterion(output, y_train_tensor)
        loss.backward()
        optimizer.step()

        if epoch == 1 or epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                val_preds = model(X_test_tensor).cpu().numpy().flatten()
                val_true = y_test_tensor.cpu().numpy().flatten()
                rmse = mean_squared_error(val_true, val_preds, squared=False)
                r2 = r2_score(val_true, val_preds)
                mape = mean_absolute_percentage_error(val_true, val_preds) * 100

                print(f"Epoch {epoch:04d} | Loss: {loss.item():.4f} | RMSE: {rmse:.2f} | R²: {r2:.4f} | MAPE: {mape:.2f}%")

                if rmse < best_rmse:
                    best_rmse = rmse
                    torch.save(model.state_dict(), model_path)
                    print(f"💾 Best model saved at epoch {epoch} (RMSE: {rmse:.2f})")

    # 🔁 Reload best model before final prediction
    model.load_state_dict(torch.load(model_path))
    model.eval()

    X_final_tensor = torch.tensor(test_final, dtype=torch.float32).to(device)
    with torch.no_grad():
        final_preds = model(X_final_tensor).cpu().numpy().flatten()
        final_preds = np.clip(final_preds, 0, None)

    # Save submission
    submission = pd.DataFrame({
        'Item_Identifier': test_ids['Item_Identifier'].values,
        'Outlet_Identifier': test_ids['Outlet_Identifier'].values,
        'Item_Outlet_Sales': final_preds
    })

    submission.to_csv(submission_file, index=False)
    print(f"✅ Submission saved to: {submission_file}")
    print(f"📦 Best model weights saved to: {model_path}")


In [132]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [133]:
gc.collect()
torch.cuda.empty_cache()


In [134]:
import torch
torch.cuda.empty_cache()

In [135]:
evaluate_and_predict_final(
    X_train, X_test, y_train, y_test,
    test_final=test[predictors],
    test_ids=test[['Item_Identifier', 'Outlet_Identifier']]
)


🔥 Training PyTorch MLP...
Epoch 0001 | Loss: 7842857.0000 | RMSE: 2673.48 | R²: -1.5906 | MAPE: 99.96%
💾 Best model saved at epoch 1 (RMSE: 2673.48)
Epoch 0010 | Loss: 4742855.0000 | RMSE: 2086.57 | R²: -0.5780 | MAPE: 270.21%
💾 Best model saved at epoch 10 (RMSE: 2086.57)
Epoch 0020 | Loss: 3285808.5000 | RMSE: 2036.82 | R²: -0.5037 | MAPE: 171.10%
💾 Best model saved at epoch 20 (RMSE: 2036.82)
Epoch 0030 | Loss: 3019185.2500 | RMSE: 1675.83 | R²: -0.0179 | MAPE: 152.78%
💾 Best model saved at epoch 30 (RMSE: 1675.83)
Epoch 0040 | Loss: 2758077.5000 | RMSE: 1607.87 | R²: 0.0630 | MAPE: 173.83%
💾 Best model saved at epoch 40 (RMSE: 1607.87)
Epoch 0050 | Loss: 2426232.2500 | RMSE: 1501.09 | R²: 0.1833 | MAPE: 137.39%
💾 Best model saved at epoch 50 (RMSE: 1501.09)
Epoch 0060 | Loss: 2203018.2500 | RMSE: 1491.75 | R²: 0.1934 | MAPE: 113.40%
💾 Best model saved at epoch 60 (RMSE: 1491.75)
Epoch 0070 | Loss: 2031223.5000 | RMSE: 1373.33 | R²: 0.3164 | MAPE: 106.59%
💾 Best model saved at epoch

Epoch 0950 | Loss: 1176650.7500 | RMSE: 1143.67 | R²: 0.5259 | MAPE: 47.26%
Epoch 0960 | Loss: 1181176.7500 | RMSE: 1258.92 | R²: 0.4256 | MAPE: 45.50%
Epoch 0970 | Loss: 1170875.3750 | RMSE: 1154.97 | R²: 0.5165 | MAPE: 46.74%
Epoch 0980 | Loss: 1164911.5000 | RMSE: 1197.52 | R²: 0.4802 | MAPE: 45.75%
Epoch 0990 | Loss: 1160203.2500 | RMSE: 1186.81 | R²: 0.4895 | MAPE: 45.81%
Epoch 1000 | Loss: 1164831.6250 | RMSE: 1175.28 | R²: 0.4994 | MAPE: 46.27%
Epoch 1010 | Loss: 1170745.0000 | RMSE: 1161.88 | R²: 0.5107 | MAPE: 47.06%
Epoch 1020 | Loss: 1169717.2500 | RMSE: 1234.82 | R²: 0.4473 | MAPE: 45.46%
Epoch 1030 | Loss: 1163270.5000 | RMSE: 1191.68 | R²: 0.4853 | MAPE: 45.78%
Epoch 1040 | Loss: 1164989.0000 | RMSE: 1210.82 | R²: 0.4686 | MAPE: 45.54%
Epoch 1050 | Loss: 1165050.1250 | RMSE: 1184.61 | R²: 0.4914 | MAPE: 45.89%
Epoch 1060 | Loss: 1170838.3750 | RMSE: 1150.75 | R²: 0.5200 | MAPE: 47.02%
Epoch 1070 | Loss: 1158922.5000 | RMSE: 1153.80 | R²: 0.5175 | MAPE: 46.52%
Epoch 1080 |

In [137]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

# ---------------------- MLP Definition ----------------------
import torch.nn as nn

class TorchMLP(nn.Module):
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(),

            nn.Linear(4096, 2048),
            nn.ReLU(),

            nn.Linear(2048, 1024),
            nn.ReLU(),

            nn.Linear(1024, 768),
            nn.ReLU(),

            nn.Linear(768, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),  # Dropout added here (deeper stage)

            nn.Linear(256, 128),
            nn.ReLU(),

    
            nn.Linear(128, 64),
            nn.Dropout(dropout_rate), 
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)



# ---------------------- Training + Prediction ----------------------
def evaluate_and_predict_final(X_train, X_test, y_train, y_test, test_final, test_ids, submission_file="submission.csv", model_path="best_model.pt"):
    """
    Trains PyTorch MLP on X_train/y_train, evaluates on X_test/y_test.
    Saves best model (lowest RMSE) to model_path.
    Predicts on test_final and saves submission to CSV with test_ids.
    """
    # Ensure numpy
    X_train = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
    X_test = X_test.values if isinstance(X_test, pd.DataFrame) else X_test
    y_train = y_train.values if isinstance(y_train, pd.Series) else y_train
    y_test = y_test.values if isinstance(y_test, pd.Series) else y_test
    test_final = test_final.values if isinstance(test_final, pd.DataFrame) else test_final

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)

    model = TorchMLP(X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_rmse = float('inf')

    print("🔥 Training PyTorch MLP...")
    for epoch in range(1, 2000):
        model.train()
        optimizer.zero_grad()
        output = model(X_train_tensor)
        loss = criterion(output, y_train_tensor)
        loss.backward()
        optimizer.step()

        if epoch == 1 or epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                val_preds = model(X_test_tensor).cpu().numpy().flatten()
                val_true = y_test_tensor.cpu().numpy().flatten()
                rmse = mean_squared_error(val_true, val_preds, squared=False)
                r2 = r2_score(val_true, val_preds)
                mape = mean_absolute_percentage_error(val_true, val_preds) * 100

                print(f"Epoch {epoch:04d} | Loss: {loss.item():.4f} | RMSE: {rmse:.2f} | R²: {r2:.4f} | MAPE: {mape:.2f}%")

                if rmse < best_rmse:
                    best_rmse = rmse
                    torch.save(model.state_dict(), model_path)
                    print(f"💾 Best model saved at epoch {epoch} (RMSE: {rmse:.2f})")

    # 🔁 Reload best model before final prediction
    model.load_state_dict(torch.load(model_path))
    model.eval()

    X_final_tensor = torch.tensor(test_final, dtype=torch.float32).to(device)
    with torch.no_grad():
        final_preds = model(X_final_tensor).cpu().numpy().flatten()
        final_preds = np.clip(final_preds, 0, None)

    # Save submission
    submission = pd.DataFrame({
        'Item_Identifier': test_ids['Item_Identifier'].values,
        'Outlet_Identifier': test_ids['Outlet_Identifier'].values,
        'Item_Outlet_Sales': final_preds
    })

    submission.to_csv(submission_file, index=False)
    print(f"✅ Submission saved to: {submission_file}")
    print(f"📦 Best model weights saved to: {model_path}")


In [ ]:
evaluate_and_predict_final(
    X_train, X_test, y_train, y_test,
    test_final=test[predictors],
    test_ids=test[['Item_Identifier', 'Outlet_Identifier']]
)

🔥 Training PyTorch MLP...
Epoch 0001 | Loss: 7842988.0000 | RMSE: 2673.70 | R²: -1.5911 | MAPE: 99.99%
💾 Best model saved at epoch 1 (RMSE: 2673.70)
Epoch 0010 | Loss: 7492892.5000 | RMSE: 2427.47 | R²: -1.1358 | MAPE: 82.99%
💾 Best model saved at epoch 10 (RMSE: 2427.47)
Epoch 0020 | Loss: 3669786.2500 | RMSE: 1612.56 | R²: 0.0575 | MAPE: 130.73%
💾 Best model saved at epoch 20 (RMSE: 1612.56)
Epoch 0030 | Loss: 3405116.5000 | RMSE: 1701.19 | R²: -0.0490 | MAPE: 108.09%
Epoch 0040 | Loss: 2912135.2500 | RMSE: 1656.68 | R²: 0.0052 | MAPE: 135.94%
Epoch 0050 | Loss: 2651058.7500 | RMSE: 1572.77 | R²: 0.1034 | MAPE: 156.25%
💾 Best model saved at epoch 50 (RMSE: 1572.77)
Epoch 0060 | Loss: 2396435.0000 | RMSE: 1494.19 | R²: 0.1908 | MAPE: 126.85%
💾 Best model saved at epoch 60 (RMSE: 1494.19)
Epoch 0070 | Loss: 2202409.2500 | RMSE: 1495.16 | R²: 0.1897 | MAPE: 106.38%
Epoch 0080 | Loss: 1989972.0000 | RMSE: 1393.26 | R²: 0.2964 | MAPE: 92.73%
💾 Best model saved at epoch 80 (RMSE: 1393.26)
